In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np 
import stackview

from codex.data.codex_dataset import CodexDataset
from codex.preprocessing.illumination import Basic
from codex.preprocessing.nodes import IlluminationCorrectionNode

2025-08-10 15:01:49.706310: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754830909.719963 2924399 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754830909.724239 2924399 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-10 15:01:49.740805: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/antonio/anaconda3/envs/codex/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask

In [2]:
out_dir = "../../outputs"

node = IlluminationCorrectionNode(
    Basic(
        get_darkfield=True,
        smoothness_flatfield=1.0,
        smoothness_darkfield=1.0,
        max_iterations=500,
        max_reweight_iterations=25,
        optimization_tol=1e-6,
        optimization_tol_diff=1e-2,
        reweighting_tol=1e-3,
    ),
    CodexDataset(
        root_dir=Path("../../outputs/edof"),
        mode="raw",
        lazy_loading=False,
        read_markers=False,
    ),
    4,
    out_dir,
)

node.run()

2025-08-10 15:01:55.451352: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754830915.465152 2924620 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754830915.469286 2924620 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-10 15:01:55.523018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754830915.536929 2924621 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-10 15:01:55.539059: E external/local_xla/xla

In [4]:
ds = CodexDataset(
        root_dir=Path("../../outputs/edof"),
        mode="raw",
        lazy_loading=False,
        read_markers=False,
    )

ds.group_tiles()

img = ds[0]["img"]
img.shape

(42, 2048, 2048)

In [5]:
from codex.preprocessing.m2stitch import stitch_images
from itertools import chain, repeat

n = ds.meta.height_tiles(1)  # Number of rows (height)
m = ds.meta.width_tiles(1)  # Number of columns (width)
overlap_percentage = ds.meta.tile_overlap

stitch_model = "../../outputs/result.pkl"

# Row coordinates: each row index is repeated m times
rows = list(chain.from_iterable(repeat(row, m) for row in range(n)))

# Column coordinates: snake pattern for each row, going back and forth
cols = list(chain.from_iterable(range(m) if row % 2 == 0 else range(m - 1, -1, -1) for row in range(n)))
result_df, _ = stitch_images(
    img,
    rows,
    cols,
    overlap_diff_threshold=10,
    initial_ncc_threshold=0,
    overlap_percentage=overlap_percentage,
    pou=18,
    max_cores=16,
    use_gpu=True,
)
result_df.to_pickle(stitch_model)

Using GPU: b'NVIDIA A40'


25it [00:20,  1.64it/s]/home/antonio/projects/codex/src/codex/preprocessing/m2stitch/_translation_computation.py:61: RuntimeWarning: invalid value encountered in scalar divide
  return n / d
26it [00:21,  1.55it/s]/home/antonio/projects/codex/src/codex/preprocessing/m2stitch/_translation_computation.py:61: RuntimeWarning: invalid value encountered in scalar divide
  return n / d
34it [00:24,  3.02it/s]/home/antonio/projects/codex/src/codex/preprocessing/m2stitch/_translation_computation.py:61: RuntimeWarning: invalid value encountered in scalar divide
  return n / d
36it [00:25,  3.13it/s]/home/antonio/projects/codex/src/codex/preprocessing/m2stitch/_translation_computation.py:61: RuntimeWarning: invalid value encountered in scalar divide
  return n / d
40it [00:26,  4.66it/s]/home/antonio/projects/codex/src/codex/preprocessing/m2stitch/_translation_computation.py:61: RuntimeWarning: invalid value encountered in scalar divide
  return n / d
42it [00:27,  1.51it/s]
7it [00:05,  1.36it/s

Ready to go!


100%|██████████| 42/42 [00:03<00:00, 10.99it/s]


In [6]:
import pandas as pd

result_df = pd.read_pickle(stitch_model)

result_df["y_pos2"] = result_df["y_pos"] - result_df["y_pos"].min()
result_df["x_pos2"] = result_df["x_pos"] - result_df["x_pos"].min()

size_y = img.shape[1]
size_x = img.shape[2]

stitched_image_size = (
    result_df["y_pos2"].max() + size_y,
    result_df["x_pos2"].max() + size_x,
)
stitched_image = np.zeros_like(img, shape=stitched_image_size)
for i, row in result_df.iterrows():
    stitched_image[
        row["y_pos2"] : row["y_pos2"] + size_y,
        row["x_pos2"] : row["x_pos2"] + size_x,
    ] = img[i]

print(stitched_image.shape)
stackview.histogram(stitched_image, zoom_factor=0.06)

(13514, 11625)


In [8]:
ds = CodexDataset(
        root_dir=Path("../../outputs/illumination_correction/"),
        mode="raw",
        lazy_loading=False,
        read_markers=False,
    )

ds.group_tiles()

res = ds[0]["img"]
res.shape

(42, 2048, 2048)

In [9]:
result_df = pd.read_pickle(stitch_model)
result_df["y_pos2"] = result_df["y_pos"] - result_df["y_pos"].min()
result_df["x_pos2"] = result_df["x_pos"] - result_df["x_pos"].min()

size_y = res.shape[1]
size_x = res.shape[2]

stitched_image_size = (
    result_df["y_pos2"].max() + size_y,
    result_df["x_pos2"].max() + size_x,
)
stitched_image_corrected = np.zeros_like(res, shape=stitched_image_size)
for i, row in result_df.iterrows():
    stitched_image_corrected[
        row["y_pos2"] : row["y_pos2"] + size_y,
        row["x_pos2"] : row["x_pos2"] + size_x,
    ] = res[i]
stackview.histogram(stitched_image_corrected, zoom_factor=0.06)

In [10]:
from codex.utils.img_utils import resize

factor = 0.07
stackview.curtain(resize(stitched_image,factor=factor), resize(stitched_image_corrected, factor=factor))